# RIPE with a lightweight backbone: final phase (training, evaluation, ablations)

DLCV course project, IIIT Sri City. Vanya Awasthi, Sai Greeshma Venkatanagari.

This notebook does the **remaining work** listed in Section 8 of the mid-term report:

1. **Training under an equal budget**: VGG-19 (original RIPE) and ResNet-18 (ours) are trained with the *same* data, steps, batch size, schedule, seed and precision.
2. **Evaluation of matching performance**: both are evaluated on HPatches and MegaDepth-1500 with the same `ripe+NN` Glue Factory protocol used for the reproduction, and accuracy is related to cost.
3. **Alternative designs (ablations)**: ResNet-18 + `layer4`, narrower stem (16/32 ch), and no ImageNet initialisation.

### What exactly is compared

| ID | Model | Trained by us? | Role in the comparison |
|---|---|---|---|
| `official_vgg19_80k` | VGG-19, official weights (80k steps, ~3 days A100) | no | Upper reference. Shows what a full-budget run reaches. **Not** a fair comparison partner (~13x more training pairs at the default budget). |
| `vgg19` | VGG-19, retrained by us, budget *B* | yes | **Fair baseline.** |
| `resnet18` | ResNet-18 + full-res stem (ours), budget *B* | yes | **Main proposal.** `vgg19` vs `resnet18` is the core result. |
| `resnet18_layer4` | + layer4 (stride 32, 1024-d hypercolumn) | yes | Ablation: does deeper context help? |
| `resnet18_narrowstem` | stem 16/32 ch (480-d hypercolumn) | yes | Ablation: memory is dominated by full-res activations (mid-term Sec. 6)? |
| `resnet18_scratch` | ResNet-18 without ImageNet weights | yes | Ablation: how much does RIPE rely on ImageNet init? |

**Metrics** (identical for all rows): HPatches homography AUC @1/3/5 px (540 pairs), MegaDepth-1500 relative-pose AUC @5/10/20° (1500 pairs), plus parameters, forward time, peak memory, and training seconds per step. During training we also log RIPE's own IMW2020-val pose AUC (200 fixed pairs) for learning curves.

**Fairness rules.** Every trained model uses the same data subset, the same number of training pairs, the same seed(s), the same precision and the same hyperparameters as `conf/train.yaml` (lr 1e-3, batch 6 x 4 accumulation, schedules scaled to the shorter run). Only `backbones=...` changes.

**Order (GPU time permitting):** `vgg19` -> `resnet18` -> evaluate both (core deliverable) -> `resnet18_layer4` -> `resnet18_narrowstem` -> `resnet18_scratch` -> second seed of `vgg19` and `resnet18`.

**Documented deviations from the authors' training** (state them in the report):
* reduced budget (see Section D; the authors used 80k steps);
* a subset of the DISK MegaDepth scenes (same subset for all models);
* images pre-resized offline to the training resolution (560 px longer side, same size RIPE computes online);
* PoseLib inputs cast to float64 (compatibility fix, no change of the algorithm);
* on GPUs without bf16 (T4) we use fp16 mixed precision instead of the authors' bf16 (same for all runs).

Every cell below can be re-run after a Colab disconnect: data is cached on Drive, and training resumes from the last checkpoint.

## A. Setup (run every session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, os
cap = torch.cuda.get_device_capability()
print(torch.cuda.get_device_name(), "compute capability", cap, "| CPUs:", os.cpu_count())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/ripe_final')   # everything persistent goes here
RUNS, EVAL, FIGS = DRIVE / 'runs', DRIVE / 'eval', DRIVE / 'figures'
DATA_CACHE = DRIVE / 'data' / 'disk-data-560.tar'
for d in (RUNS, EVAL, FIGS, DATA_CACHE.parent):
    d.mkdir(parents=True, exist_ok=True)

Our GitHub repo is **private**, so Colab needs a way in. Use one of:

* **Option 1 (recommended): GitHub token.** On GitHub: Settings -> Developer settings -> Personal access tokens ->
  Fine-grained token, repository access *only* `RIPE-dlcv`, permission *Contents: Read-only*.
  In Colab: key icon (Secrets) in the left sidebar -> add secret `GH_TOKEN` with the token, enable notebook access.
  The token is never printed or written into the notebook.
* **Option 2: zip on Drive.** Zip the repo folder on your laptop (it must contain `.git` or `pyproject.toml`
  at the top, `ripe/train.py` relies on it) and upload it as `MyDrive/ripe_final/RIPE-dlcv.zip`.
  Re-upload after every code change.

In [ ]:
# our repository (upstream RIPE + our backbone, training changes and scripts)
REPO = 'github.com/sheisstarwithoutmoon/RIPE-dlcv'
REPO_ZIP = DRIVE / 'RIPE-dlcv.zip'
%cd /content
import subprocess
if not Path('/content/RIPE').exists():
    try:
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
    except Exception:
        token = None
    if token:
        # token only in the subprocess call; the remote URL is reset afterwards so it is not stored in .git/config
        subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{token}@{REPO}', 'RIPE'], check=True)
        subprocess.run(['git', '-C', 'RIPE', 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
    elif REPO_ZIP.exists():
        !mkdir -p /tmp/repo_zip && unzip -q -o {REPO_ZIP} -d /tmp/repo_zip
        # the zip may contain the folder itself or its contents
        src = next(p.parent for p in Path('/tmp/repo_zip').rglob('ripe/train.py')).parent
        !cp -r {src} /content/RIPE
        # ripe/train.py finds the project root via .git or pyproject.toml; a zip without .git needs the marker
        ![ -e /content/RIPE/.git ] || touch /content/RIPE/pyproject.toml
    else:
        raise RuntimeError('Add the GH_TOKEN secret or upload RIPE-dlcv.zip to Drive (see the cell above).')
elif Path('/content/RIPE/.git').exists():
    try:
        from google.colab import userdata
        token = userdata.get('GH_TOKEN')
        subprocess.run(['git', '-C', '/content/RIPE', 'pull', '-q', f'https://x-access-token:{token}@{REPO}'], check=False)
    except Exception:
        pass
%cd /content/RIPE
!ls ripe/models/backbones/resnet.py scripts/prepare_disk_data.py

In [ ]:
# training dependencies (conda_env.yml without conda). poselib from PyPI has prebuilt wheels.
!pip install -q lightning hydra-core pyrootutils kornia wandb poselib h5py rich torchmetrics

In [ ]:
os.environ['DATA_DIR'] = '/content/data'          # conf/data/*.yaml read ${DATA_DIR}/disk-data
os.environ['OUTPUT_DIR'] = str(RUNS)
os.environ['SLURM_JOB_ID'] = 'colab'              # referenced by the default hydra run dir
os.environ['WANDB_MODE'] = 'disabled'             # we log to CSV instead (train_metrics.csv)
os.environ['TORCH_HOME'] = str(DRIVE / 'torch_cache')   # ImageNet weights downloaded once

## B. Training data (DISK MegaDepth subset + IMW2020-val)

First session: `scripts/prepare_disk_data.py` streams the scene tarballs, keeps only the images, resizes them to the
training resolution and writes a filtered `dataset.json`; the result is cached as one tar on Drive.
Later sessions only copy and unpack the cache (a few minutes).

`BUDGET_GB` is the *download* volume (full set ~170 GB, a few GB after resizing). 40 GB is roughly 35-45 scenes and gives
far more pairs than a reduced run can consume. Download speed decides how long this takes; it is resumable.

In [ ]:
BUDGET_GB = 40
if DATA_CACHE.exists():
    !mkdir -p /content/data && tar -xf {DATA_CACHE} -C /content/data
else:
    !python scripts/prepare_disk_data.py --root /content/data/disk-data --budget-gb {BUDGET_GB} --workers {os.cpu_count()}
    !tar -cf {DATA_CACHE} -C /content/data disk-data
!du -sh /content/data/disk-data/megadepth /content/data/disk-data/imw2020-val
!cat /content/data/disk-data/megadepth/scenes_used.txt | tr '\n' ' '

In [ ]:
# sanity check: one positive and one negative training pair exactly as the training loop sees them
import hydra, matplotlib.pyplot as plt
from omegaconf import OmegaConf
from hydra.utils import instantiate
cfg = OmegaConf.load('conf/data/disk_megadepth.yaml')
cfg.root = '/content/data/disk-data'
ds = instantiate(cfg)
print('pairs in dataset (pos+neg):', len(ds))
fig, ax = plt.subplots(2, 2, figsize=(10, 10))
mean, std = torch.tensor([0.485, 0.456, 0.406])[:, None, None], torch.tensor([0.229, 0.224, 0.225])[:, None, None]
for r, idx in enumerate([0, 1]):
    s = ds[idx]
    for c, k in enumerate(['src_image', 'trg_image']):
        ax[r, c].imshow((s[k] * std + mean).clamp(0, 1).permute(1, 2, 0)); ax[r, c].axis('off')
    ax[r, 0].set_title(f"label={s['label']} ({'same scene' if s['label'] else 'different scenes'})", loc='left')
plt.show()

## C. Computational cost of all variants (no training needed)

Same measurement as the mid-term (`compare_backbones.py`), now including the ablation variants. Run it on the GPU you report
(the mid-term numbers are from a T4).

In [ ]:
!python compare_backbones.py --all --json {DRIVE}/cost.json

## D. Training budget: timing probe

The budget is defined as a number of **training pairs** so it stays identical if the batch size has to change.
The authors trained 80,000 steps x 6 pairs = 480k pairs. We run a short probe of each backbone (no validation) to measure
seconds per step and peak memory, then fix the budget for all runs.

If the probe runs out of memory, set `BATCH, ACCUM = 3, 8`: optimizer updates, pairs per update (24) and all schedules stay the same,
because `num_steps` is derived from the pair budget.

In [ ]:
BATCH, ACCUM = 6, 4                     # authors' setting (24 pairs per optimizer update)
PRECISION = 'bf16-mixed' if torch.cuda.get_device_capability()[0] >= 8 else '16-mixed'
WORKERS = min(8, os.cpu_count())
print(PRECISION, BATCH, ACCUM, WORKERS)

def train_cmd(run, backbone, num_steps, val_interval, seed=32000, out=RUNS):
    return (f"RIPE_SEED={seed} python ripe/train.py --config-name train "
            f"project_name=ripe_dlcv name={run} backbones={backbone} "
            f"num_steps={num_steps} val_interval={val_interval} batch_size={BATCH} num_grad_accs={ACCUM} "
            f"precision={PRECISION} num_workers={WORKERS} wandb_mode=disabled "
            f"hydra.run.dir={out}/{run}")

In [ ]:
# ~5-10 min per backbone; probe runs go to /content (not Drive)
for bb in ['vgg', 'resnet']:
    !rm -rf /content/probe/{bb}
    cmd = train_cmd(f'probe_{bb}', bb, num_steps=60, val_interval=0, out='/content/probe')
    !{cmd} log_interval=10 ckpt_interval=0 2>&1 | tail -n 3
import pandas as pd
for bb in ['vgg', 'resnet']:
    df = pd.read_csv(f'/content/probe/probe_{bb}/train_metrics.csv')
    print(f"{bb:7s} sec/step (median, after warm-up) = {df.sec_per_step.iloc[2:].median():.2f}   peak GPU mem = {df.gpu_mem_gb.max():.1f} GB")

In [ ]:
# ---- fix the budget here (identical for every run) ----
BUDGET_PAIRS = 36_000            # 7.5% of the authors' 480k pairs = 6000 steps at batch 6
NUM_STEPS = BUDGET_PAIRS // BATCH
VAL_INTERVAL = NUM_STEPS // 10   # 11 points on the learning curve (+ final)
print(f"num_steps={NUM_STEPS}, val every {VAL_INTERVAL} steps")
# rough wall-clock estimate per run: NUM_STEPS * sec/step (VGG is the slower one) + ~11 validations

## E. Training runs

Each cell trains one model. If Colab disconnects, **run the setup cells (A, B) and the budget cell (D) again, then re-run the same training cell**:
it resumes from `checkpoints/last.pth` (saved every 250 steps on Drive). A run is complete when `model_<steps>_final.pth` exists.

Logs: `train_metrics.csv` (reward, keypoints, losses, sec/step, memory), `val_metrics.csv` (IMW2020-val AUC), `backbone.yaml` (needed for evaluation).

### E1. VGG-19 baseline (core)

In [ ]:
run = 'vgg19_s32000'
if not list((RUNS / run).glob('model_*_final.pth')):
    !{train_cmd(run, 'vgg', NUM_STEPS, VAL_INTERVAL, seed=32000)}
else:
    print(run, 'already finished')

### E2. ResNet-18, ours (core)

In [ ]:
run = 'resnet18_s32000'
if not list((RUNS / run).glob('model_*_final.pth')):
    !{train_cmd(run, 'resnet', NUM_STEPS, VAL_INTERVAL, seed=32000)}
else:
    print(run, 'already finished')

### E3. Ablation: ResNet-18 + layer4

In [ ]:
run = 'resnet18_layer4_s32000'
if not list((RUNS / run).glob('model_*_final.pth')):
    !{train_cmd(run, 'resnet_layer4', NUM_STEPS, VAL_INTERVAL, seed=32000)}
else:
    print(run, 'already finished')

### E4. Ablation: narrow stem (16/32 ch)

In [ ]:
run = 'resnet18_narrowstem_s32000'
if not list((RUNS / run).glob('model_*_final.pth')):
    !{train_cmd(run, 'resnet_narrowstem', NUM_STEPS, VAL_INTERVAL, seed=32000)}
else:
    print(run, 'already finished')

### E5. Ablation: no ImageNet initialisation

In [ ]:
run = 'resnet18_scratch_s32000'
if not list((RUNS / run).glob('model_*_final.pth')):
    !{train_cmd(run, 'resnet_scratch', NUM_STEPS, VAL_INTERVAL, seed=32000)}
else:
    print(run, 'already finished')

### E6. Second seed: VGG-19

In [ ]:
run = 'vgg19_s1'
if not list((RUNS / run).glob('model_*_final.pth')):
    !{train_cmd(run, 'vgg', NUM_STEPS, VAL_INTERVAL, seed=1)}
else:
    print(run, 'already finished')

### E7. Second seed: ResNet-18

In [ ]:
run = 'resnet18_s1'
if not list((RUNS / run).glob('model_*_final.pth')):
    !{train_cmd(run, 'resnet', NUM_STEPS, VAL_INTERVAL, seed=1)}
else:
    print(run, 'already finished')

## F. Evaluation on HPatches and MegaDepth-1500

Same Glue Factory fork and `ripe+NN` protocol as the reproduction (Section 4 of the mid-term report).
`scripts/setup_gluefactory_ckpt.py` points the fork's RIPE submodule at our repository and adds a config `ripe_ckpt+NN`
that is a copy of `ripe+NN` with only the model loader changed, plus the PoseLib fix. So the official weights and every
checkpoint are evaluated with identical resolution, keypoint count, matcher and RANSAC settings.

In [ ]:
%cd /content
!pip install -q uv
!apt-get install -y -q libeigen3-dev > /dev/null
![ -d glue-factory ] || git clone https://github.com/JohannesK14/glue-factory
%cd /content/glue-factory
!git submodule update --init --recursive
!uv sync
!uv pip install pyyaml
!python /content/RIPE/scripts/setup_gluefactory_ckpt.py /content/glue-factory /content/RIPE
%cd /content/RIPE

In [ ]:
# official weights (re-evaluated in this environment, so all rows share the same software stack)
!wget -q --no-check-certificate https://cvg.hhi.fraunhofer.de/RIPE/ripe_weights.pth -O /tmp/ripe_weights.pth
!python scripts/eval_gluefactory.py --tag official_vgg19_80k --official --out {EVAL}

In [ ]:
# every finished run (skips runs that are already evaluated)
for run_dir in sorted(RUNS.iterdir()):
    ckpts = sorted(run_dir.glob('model_*_final.pth'))
    if ckpts:
        !python scripts/eval_gluefactory.py --tag {run_dir.name} --ckpt {ckpts[-1]} --out {EVAL}

## G. Qualitative comparison (example pair, protocol of mid-term Figure 2)

In [ ]:
models = ['--model "VGG-19 official (80k steps)=official"']
for variant, label in [('vgg19', 'VGG-19 (retrained)'), ('resnet18', 'ResNet-18 (ours)')]:
    ck = sorted((RUNS / f'{variant}_s32000').glob('model_*_final.pth'))
    if ck:
        models.append(f'--model "{label}={ck[-1]}"')
!python scripts/qualitative.py {' '.join(models)} --out {FIGS}/fig_qualitative.png
from IPython.display import Image, display
display(Image(f'{FIGS}/fig_qualitative.png', width=900))

## H. Figures and result table

`scripts/plot_results.py` reads everything above from Drive and writes to `figures/`:

| File | Shows |
|---|---|
| `fig_training_curves.png` | mean reward, selected keypoints, descriptor loss, skipped pairs vs. step (all variants) |
| `fig_val_auc.png` | IMW2020-val AUC@5/10/20 during training |
| `fig_hpatches_auc.png`, `fig_megadepth1500_auc.png` | final AUCs of all models (official = grey reference) |
| `fig_cumulative_errors.png` | cumulative error curves on both benchmarks |
| `fig_cost_relative.png` | params / time / memory of all variants relative to VGG-19 |
| `fig_accuracy_vs_cost.png` | the trade-off the project is about: accuracy vs. forward time |
| `results_table.md/.tex` | every number in one table (mean +- std when two seeds exist) |

In [ ]:
!python scripts/plot_results.py --runs-dir {RUNS} --eval-dir {EVAL} --cost-json {DRIVE}/cost.json --out {FIGS} --budget "{NUM_STEPS} steps x {BATCH} pairs"
from IPython.display import Image, display, Markdown
for f in ['fig_training_curves', 'fig_val_auc', 'fig_hpatches_auc', 'fig_megadepth1500_auc',
          'fig_cumulative_errors', 'fig_cost_relative', 'fig_accuracy_vs_cost']:
    p = FIGS / f'{f}.png'
    if p.exists():
        display(Markdown(f'**{f}**')); display(Image(str(p), width=850))
display(Markdown((FIGS / 'results_table.md').read_text()))

## I. How to read the results (for the final report)

* **Core claim** = `vgg19` vs `resnet18` at the same budget. Report the AUC difference on both benchmarks together with the 3.7x / 2.7x cost reduction. Only claim a difference larger than the seed-to-seed spread (E6/E7).
* `official_vgg19_80k` shows how far the reduced budget is from full training. Both of our models will be below it; that is expected and not a weakness of the backbone.
* `fig_val_auc.png`: if the curves are still rising at the end, the ranking may change with longer training; say so.
* `resnet18_layer4`: better accuracy -> deep context matters; similar accuracy -> layer4 is not worth its 9.5 M parameters.
* `resnet18_narrowstem`: compare its peak memory in `cost.json` with `resnet18` (tests the mid-term explanation) and its accuracy (does the stem need 64 channels?).
* `resnet18_scratch`: the gap to `resnet18` measures how much RIPE's weak reward signal relies on ImageNet features.
* The training-curve panels (reward, selected keypoints, skipped pairs) show whether REINFORCE was stable for each backbone, which tests the "optimisation stability" hypothesis of the mid-term report (Section 5.2).